# Why the score was too good

MichAl Academy, lesson 2.3.

Run each cell with **Shift+Enter**.

Four experiments. In three of them a model reports an excellent score and has
learned nothing useful, and in the fourth the famous warning turns out to make
no measurable difference at all. Knowing which is which is the point.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def accuracy(X, y):
    """Mean five-fold accuracy of a scaled logistic regression."""
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    return cross_val_score(model, X, y, cv=CV).mean()


## 1. A feature that records the answer

Lesson 2.2 measured a feature that was worth almost nothing: the total amount of
ink in a handwritten digit. Use it here on the hardest pair to tell apart, 3
against 8, so we start from something honestly close to useless.


In [ ]:
digits = load_digits()
pair = np.isin(digits.target, [3, 8])
ink = digits.data[pair].sum(axis=1).reshape(-1, 1)
y = (digits.target[pair] == 8).astype(int)

print(f"{len(y)} images, {y.mean():.1%} of them are 8s")
print(f"total ink alone:  {accuracy(ink, y):.3f}")


Barely better than answering "3" every time.

Now add one column. Imagine a reviewer looked at each image and ticked a box
saying it was an 8, and imagine they were right 97% of the time. That column
goes into the table alongside the ink.


In [ ]:
rng = np.random.default_rng(0)
reviewed = np.where(rng.random(len(y)) < 0.97, y, 1 - y).reshape(-1, 1)

X_leak = np.hstack([ink, reviewed])
print(f"total ink + reviewer's tick:  {accuracy(X_leak, y):.3f}")


From 0.616 to 0.955 on a feature set we already established is worthless. No
model got better at reading handwriting.

That column is not a clue about the answer. It **is** the answer, recorded by
somebody who already knew it. Which raises the only question that matters about
any feature: would this value exist at the moment you need the prediction?


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y, test_size=0.3, random_state=0, stratify=y
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
model.fit(X_train, y_train)

print(f"held back, reviewer's tick available:   {model.score(X_test, y_test):.3f}")

# In production the image arrives before anyone has looked at it.
X_prod = X_test.copy()
X_prod[:, -1] = 0
print(f"held back, nobody has reviewed it yet:  {model.score(X_prod, y_test):.3f}")


A coin flip. Every point above chance came from a column that does not exist
when a prediction is actually needed.

Look at what the model decided to rely on:


In [ ]:
weights = np.abs(model[-1].coef_.ravel())
print(f"weight on total ink        {weights[0]:.3f}")
print(f"weight on reviewer's tick  {weights[1]:.3f}")
print(f"ratio                      {weights[1] / weights[0]:.0f}x")


This is target leakage, and the Track 1 capstone had one planted in it:
`reported_by_user`, non-zero for every phishing row and zero for every benign
one. Same shape, same cause, same cure. Ask of every column: is this a property
of the thing, or a record of what somebody concluded about it?


## 2. Choosing features before you validate

The second kind of leak needs no bad column at all. It comes from doing a
legitimate step in the wrong order.

Build a dataset with **no signal whatsoever**: 200 rows, 5,000 columns of pure
noise, and labels that are coin flips unrelated to any of it. The only honest
score on this data is 0.500.


In [ ]:
noise_rng = np.random.default_rng(0)
N, P, KEEP = 200, 5000, 20

X_noise = noise_rng.normal(size=(N, P))
y_coin = noise_rng.integers(0, 2, size=N)

print("shape:", X_noise.shape)
print("correlation of any column with the label is chance alone")


In [ ]:
# The wrong order: find the 20 most promising columns using every row, then
# cross-validate a model on those columns.
promising = SelectKBest(f_classif, k=KEEP).fit(X_noise, y_coin).get_support()
wrong = cross_val_score(
    LogisticRegression(max_iter=2000), X_noise[:, promising], y_coin, cv=CV
).mean()

# The right order: selection happens inside each fold, using only that fold's
# training rows. A Pipeline is what enforces it.
pipeline = make_pipeline(SelectKBest(f_classif, k=KEEP), LogisticRegression(max_iter=2000))
right = cross_val_score(pipeline, X_noise, y_coin, cv=CV).mean()

print(f"select on all the data, then validate:  {wrong:.3f}")
print(f"select inside each fold:                {right:.3f}")
print(f"the truth:                              0.500")


Read that again. The data contains nothing at all, and the first procedure
reports accuracy well into the seventies.

Nothing was faked. With 5,000 noise columns and 200 rows, some columns will
match the labels closely by luck. Choosing them while looking at every row means
the choice already knows the test rows' answers, so the held-back fold is no
longer held back.

The second number is what honest procedure looks like on data with no signal:
close to 0.500, and the small excess is what 200 rows of noise is worth.

The scikit-learn pitfalls guide states the general rule: leakage "occurs when
information that would not be available at prediction time is used when building
the model", and always "split the data into train and test subsets first,
particularly before any preprocessing steps". A `Pipeline` is how you make that
structural rather than remembered.


## 3. The same row on both sides

Third kind. Nobody chose a bad feature and nobody reordered anything: the table
simply contains rows more than once, and a random split puts copies of one row
in both halves.

Watch what the inflation depends on.


In [ ]:
def holdout(X, y, seed):
    a, b, c, d = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(a, c).score(b, d)


def average_holdout(X, y):
    return np.mean([holdout(X, y, s) for s in range(5)])


def with_duplicates(X, y, seed=0):
    r = np.random.default_rng(seed)
    copies = r.choice(len(X), len(X), replace=True)     # one extra copy per row, on average
    return np.vstack([X, X[copies]]), np.hstack([y, y[copies]])


print(f"{'rows':>6} {'clean':>7} {'duplicated':>11} {'inflation':>10}")
for n in (150, 300, 600, 1797):
    keep = np.random.default_rng(1).choice(len(digits.data), n, replace=False)
    Xs, ys = digits.data[keep], digits.target[keep]
    clean = average_holdout(Xs, ys)
    dirty = average_holdout(*with_duplicates(Xs, ys))
    print(f"{n:>6} {clean:>7.3f} {dirty:>11.3f} {dirty - clean:>+10.3f}")


The inflation is largest where the dataset is smallest, and it shrinks as the
data grows. That is not a coincidence and it is worth understanding rather than
memorising.

Duplicate rows buy the model exactly as much as memorising is worth. With 150
rows the model cannot generalise well, so recognising a row it has already seen
is a big win. With 1,797 rows it already scores 0.964 honestly, so there is
barely any headroom left to steal.

Which means this leak is worst precisely when a dataset is small, which is
exactly when people are most tempted to pad it.


## 4. The warning that made no difference here

The advice you will meet most often is to fit your scaler on the training rows
only, never on the whole table. It is correct: the test rows' values influence
the mean and the spread, so information crosses the line.

Measure it.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=0, stratify=digits.target
)

fitted_on_everything = StandardScaler().fit(digits.data)
leaky = LogisticRegression(max_iter=3000).fit(fitted_on_everything.transform(X_tr), y_tr)

fitted_on_train = StandardScaler().fit(X_tr)
clean = LogisticRegression(max_iter=3000).fit(fitted_on_train.transform(X_tr), y_tr)

print(f"scaler fitted on the whole table:  {leaky.score(fitted_on_everything.transform(X_te), y_te):.4f}")
print(f"scaler fitted on training rows:    {clean.score(fitted_on_train.transform(X_te), y_te):.4f}")


Identical to four decimal places.

Report that honestly rather than pretending otherwise. A mean and a standard
deviation computed over 1,797 rows barely move when you add or remove 539 of
them, so the leak is real and its effect here is nil.

It is worth doing anyway, for two reasons that are not "because the numbers say
so". Small test sets move those statistics much more. And the same mistake with
a more aggressive step, imputing missing values, encoding a category by its
average label, or resampling to balance classes, moves the score a great deal.
The habit costs nothing and the failure mode is silent, which is the only
argument it needs.


## What to take from this

| Kind of leak | What happened | Measured |
|---|---|---|
| Target leakage | A column recorded the answer | 0.616 to 0.955, and 0.509 once the column is gone |
| Preprocessing before splitting | Features chosen while looking at every row | 0.795 on data containing no signal |
| Contamination | The same rows on both sides | +0.076 at 150 rows, +0.023 at 1,797 |
| Transform fitted on everything | Scaler saw the test rows | No measurable difference here |

Three habits cover all four.

**Split first, then do everything else.** Put every transform inside a
`Pipeline` so the order is enforced by the code rather than by your memory.

**Ask of every column: would this value exist at prediction time?** It is the
one question that catches target leakage, and no amount of validation will catch
it for you.

**Distrust a good score.** A result far better than the problem deserves is
evidence of a mistake far more often than it is evidence of a breakthrough. That
instinct is the most valuable thing in this lesson.
